In [2]:
import numpy as np
import pandas as pd
import faiss
import os

In [3]:
embeddings = np.load(
    "../data/embeddings/multimodal_embeddings.npy"
).astype("float32")

product_ids = pd.read_csv(
    "../data/embeddings/multimodal_product_ids.csv"
)

print("Embedding shape:", embeddings.shape)
print("Product IDs:", len(product_ids))

Embedding shape: (4996, 256)
Product IDs: 4996


In [4]:
faiss.normalize_L2(embeddings)

print("Embeddings normalized.")

Embeddings normalized.


In [5]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)

index.add(embeddings)

print("FAISS index created.")
print("Number of vectors:", index.ntotal)

FAISS index created.
Number of vectors: 4996


In [6]:
os.makedirs(
    "../index",
    exist_ok=True
)

faiss.write_index(
    index,
    "../index/product_index.faiss"
)

print("FAISS index saved successfully.")

FAISS index saved successfully.


In [7]:
import pandas as pd
import numpy as np
import faiss

products = pd.read_csv(
    "../data/processed/products.csv"
)

product_ids = pd.read_csv(
    "../data/embeddings/multimodal_product_ids.csv"
)

print("Products:", len(products))
print("FAISS IDs:", len(product_ids))

Products: 5000
FAISS IDs: 4996


In [8]:
index = faiss.read_index(
    "../index/product_index.faiss"
)

print("FAISS vectors:", index.ntotal)

FAISS vectors: 4996


In [9]:
embeddings = np.load(
    "../data/embeddings/multimodal_embeddings.npy"
).astype("float32")

print("Embedding shape:", embeddings.shape)

Embedding shape: (4996, 256)


In [10]:
query_vector = embeddings[0].reshape(1, -1)

In [11]:
scores, indices = index.search(
    query_vector,
    11
)

print("Indices:", indices[0])
print("Scores:", scores[0])

Indices: [   0 3295 1459 4525 2303 2049 3245 2525 4583  613 4343]
Scores: [1.         0.6939633  0.6737695  0.6674958  0.6624954  0.658817
 0.6580123  0.65151155 0.64945984 0.64234173 0.6365808 ]


In [12]:
results = []

query_product_id = product_ids.iloc[0]["product_id"]

for rank, idx in enumerate(indices[0]):

    product_id = product_ids.iloc[idx]["product_id"]

    if product_id == query_product_id:
        continue

    product = products[
        products["product_id"] == product_id
    ]

    if len(product) == 0:
        continue

    row = product.iloc[0].copy()

    results.append({
        "rank": len(results) + 1,
        "product_id": product_id,
        "title": row["title"],
        "price": row["price"],
        "rating": row["average_rating"],
        "similarity": float(scores[0][rank])
    })

    if len(results) == 10:
        break

recommendations = pd.DataFrame(results)

print(
    recommendations.to_string(index=False)
)

 rank product_id                                                                                                                                                                                               title  price  rating  similarity
    1 B0919TZ25H                                            YOOESTORES82 2PCS Colorful Heart Ring Gold Band Rings Set for Women Girl Couple Cute Love Heart Plain Stackable Finger Rings Wedding Preppy Jewelry Gift   9.98     3.9    0.693963
    2 B0737XSZST                                                                                                              MORE FUN Retro 12 Constellation Handmade Brown Leather Bangle Bracelet for Mens Womens  13.99     3.9    0.673769
    3 B0BPR7RPZ6                                                      LANE WOODS Mother's Day Gifts Mother Daughter Birthstone Rings for Teen Girls Women 925 Sterling Silver Double Heart Open Adjustable Band Ring  21.99     4.2    0.667496
    4 B013D3ZLDG                        

In [13]:
from sentence_transformers import SentenceTransformer

text_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Sentence-BERT loaded successfully")

d:\DL_CP\multimodal-ecommerce-recommender\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1610.08it/s]


Sentence-BERT loaded successfully


In [15]:
query = "comfortable black running shoes"

print("Query:", query)

Query: comfortable black running shoes


In [16]:
query_embedding = text_model.encode(
    [query],
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Query embedding shape:", query_embedding.shape)

Query embedding shape: (1, 384)


In [17]:
text_embeddings = np.load(
    "../data/embeddings/text_embeddings.npy"
).astype("float32")

text_product_ids = pd.read_csv(
    "../data/embeddings/text_product_ids.csv"
)

print("Text embeddings:", text_embeddings.shape)
print("Text product IDs:", len(text_product_ids))

Text embeddings: (5000, 384)
Text product IDs: 5000


In [18]:
faiss.normalize_L2(text_embeddings)

print("Text embeddings normalized")

Text embeddings normalized


In [19]:
text_index = faiss.IndexFlatIP(
    text_embeddings.shape[1]
)

text_index.add(text_embeddings)

print(
    "Text FAISS vectors:",
    text_index.ntotal
)

Text FAISS vectors: 5000


In [20]:
query = "comfortable black running shoes"

print("Query:", query)

Query: comfortable black running shoes


In [21]:
from sentence_transformers import SentenceTransformer

text_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Sentence-BERT loaded successfully")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2270.38it/s]


Sentence-BERT loaded successfully


In [22]:
query_embedding = text_model.encode(
    [query],
    convert_to_numpy=True,
    normalize_embeddings=True
)

print(
    "Query embedding shape:",
    query_embedding.shape
)

Query embedding shape: (1, 384)


In [23]:
scores, indices = text_index.search(
    query_embedding.astype("float32"),
    10
)

print("Scores:")
print(scores[0])

print("\nIndices:")
print(indices[0])

Scores:
[0.7281321  0.6913062  0.6896286  0.6671386  0.6650946  0.65877706
 0.651947   0.648363   0.64117116 0.64062655]

Indices:
[4183 2800 4579   56 1999 3824 1229 3081 4156 3666]


In [24]:
text_results = []

for rank, idx in enumerate(indices[0]):

    product_id = text_product_ids.iloc[idx]["product_id"]

    product = products[
        products["product_id"] == product_id
    ]

    if len(product) == 0:
        continue

    row = product.iloc[0]

    text_results.append({
        "rank": len(text_results) + 1,
        "product_id": product_id,
        "title": row["title"],
        "price": row["price"],
        "rating": row["average_rating"],
        "similarity": float(scores[0][rank])
    })

text_results = pd.DataFrame(text_results)

print(
    text_results.to_string(index=False)
)

 rank product_id                                                                                                                   title  price  rating  similarity
    1 B097MV5T27                                                                WsDebutting Men's Stylish Personality Lifestyle Sneakers    NaN     3.9    0.728132
    2 B00D4LILKI                                Brooks Women's Adrenaline GTS 14 Running Shoes, Color: White/Fuschia/Midnight, Size: 9.0    NaN     4.7    0.691306
    3 B00YQFDYLC                                                                                Air Balance Men Black Grey Running Shoes    NaN     4.0    0.689629
    4 B08FR4SRYS                                       Ezkrwxn Women Sport Running Shoes Fashion Casual Atheltic Walking Tennis Sneakers  39.90     4.2    0.667139
    5 B0725GW2QY                                                                                   Brooks Launch 5 Women's Running Shoes    NaN     4.6    0.665095
    6 B09VB9Z2P4

In [25]:
def recommend_from_text(
    query,
    top_k=10
):

    query_embedding = text_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = text_index.search(
        query_embedding,
        top_k
    )

    results = []

    for rank, idx in enumerate(indices[0]):

        product_id = text_product_ids.iloc[
            idx
        ]["product_id"]

        product = products[
            products["product_id"] == product_id
        ]

        if len(product) == 0:
            continue

        row = product.iloc[0]

        results.append({
            "rank": len(results) + 1,
            "product_id": product_id,
            "title": row["title"],
            "price": row["price"],
            "rating": row["average_rating"],
            "similarity": float(scores[0][rank])
        })

    return pd.DataFrame(results)

In [26]:
recommendations = recommend_from_text(
    "comfortable black running shoes",
    top_k=10
)

print(
    recommendations.to_string(index=False)
)

 rank product_id                                                                                                                   title  price  rating  similarity
    1 B097MV5T27                                                                WsDebutting Men's Stylish Personality Lifestyle Sneakers    NaN     3.9    0.728132
    2 B00D4LILKI                                Brooks Women's Adrenaline GTS 14 Running Shoes, Color: White/Fuschia/Midnight, Size: 9.0    NaN     4.7    0.691306
    3 B00YQFDYLC                                                                                Air Balance Men Black Grey Running Shoes    NaN     4.0    0.689629
    4 B08FR4SRYS                                       Ezkrwxn Women Sport Running Shoes Fashion Casual Atheltic Walking Tennis Sneakers  39.90     4.2    0.667139
    5 B0725GW2QY                                                                                   Brooks Launch 5 Women's Running Shoes    NaN     4.6    0.665095
    6 B09VB9Z2P4

In [27]:
queries = [
    "comfortable running shoes",
    "women summer dress",
    "black leather handbag",
    "formal men's shoes",
    "silver jewelry"
]

for query in queries:

    print("\n" + "=" * 70)
    print("QUERY:", query)

    result = recommend_from_text(
        query,
        top_k=5
    )

    print(
        result[
            [
                "rank",
                "title",
                "price",
                "rating",
                "similarity"
            ]
        ].to_string(index=False)
    )


QUERY: comfortable running shoes
 rank                                                                                                                                              title  price  rating  similarity
    1                                                                                           WsDebutting Men's Stylish Personality Lifestyle Sneakers    NaN     3.9    0.750971
    2                                                       M MGRNDL Womens Air Running Shoes Breathable Tennis Fashion Sneakers for Walking Jogging Gym  19.99     4.1    0.723256
    3                                              RUNMAXX Running Shoes for Men Walking Shoes Non Slip Fashion Sneakers Gym Tennis Sport Athletic Shoes  22.99     4.3    0.713353
    4 UBFEN Womens Running Shoes Fashion Sneakers Sports Casual Footwear Walking Fitness Jogging Athletic Indoor Outdoor 10.5 Women / 8.5 Men US B White    NaN     3.1    0.682078
    5                                                             

In [28]:
def filter_by_price(
    results,
    max_price
):

    result = results.copy()

    result["price_numeric"] = pd.to_numeric(
        result["price"],
        errors="coerce"
    )

    result = result[
        result["price_numeric"] <= max_price
    ]

    return result

In [29]:
recommendations = recommend_from_text(
    "comfortable running shoes",
    top_k=20
)

filtered = filter_by_price(
    recommendations,
    max_price=50
)

print(
    filtered.to_string(index=False)
)

 rank product_id                                                                                                                       title  price  rating  similarity  price_numeric
    2 B09VB9Z2P4                                M MGRNDL Womens Air Running Shoes Breathable Tennis Fashion Sneakers for Walking Jogging Gym  19.99     4.1    0.723256          19.99
    3 B09D3N4PRM                       RUNMAXX Running Shoes for Men Walking Shoes Non Slip Fashion Sneakers Gym Tennis Sport Athletic Shoes  22.99     4.3    0.713353          22.99
    7 B08K7FZ59Q Soulsfeng Running Shoes Low-top Lace-up Breathable Gym Trainer Tennis Walking Athletic Shoes Fashion Sneakers for Men Women  39.00     4.5    0.668544          39.00
   13 B0BK8YSW1V                                                                    TSIODFO Women Running Shoes Gym Jogging Walking Sneakers  19.90     3.6    0.651814          19.90
   14 B091GRYC4V                                Mens Running Shoes Slip-on Walking Sn

In [30]:
def filter_by_rating(
    results,
    min_rating
):

    result = results.copy()

    result["rating_numeric"] = pd.to_numeric(
        result["rating"],
        errors="coerce"
    )

    result = result[
        result["rating_numeric"] >= min_rating
    ]

    return result

In [31]:
filtered = filter_by_rating(
    recommendations,
    min_rating=4.0
)

print(
    filtered.to_string(index=False)
)

 rank product_id                                                                                                                                        title  price  rating  similarity  rating_numeric
    2 B09VB9Z2P4                                                 M MGRNDL Womens Air Running Shoes Breathable Tennis Fashion Sneakers for Walking Jogging Gym  19.99     4.1    0.723256             4.1
    3 B09D3N4PRM                                        RUNMAXX Running Shoes for Men Walking Shoes Non Slip Fashion Sneakers Gym Tennis Sport Athletic Shoes  22.99     4.3    0.713353             4.3
    5 B0725GW2QY                                                                                                        Brooks Launch 5 Women's Running Shoes    NaN     4.6    0.681711             4.6
    7 B08K7FZ59Q                  Soulsfeng Running Shoes Low-top Lace-up Breathable Gym Trainer Tennis Walking Athletic Shoes Fashion Sneakers for Men Women  39.00     4.5    0.668544            

In [32]:
def recommend_with_filters(
    query,
    top_k=10,
    max_price=None,
    min_rating=None
):

    results = recommend_from_text(
        query,
        top_k=top_k * 3
    )

    if max_price is not None:

        results = filter_by_price(
            results,
            max_price
        )

    if min_rating is not None:

        results = filter_by_rating(
            results,
            min_rating
        )

    return results.head(top_k)

In [33]:
final_results = recommend_with_filters(
    query="comfortable black running shoes",
    top_k=10,
    max_price=50,
    min_rating=4.0
)

print(
    final_results.to_string(index=False)
)

 rank product_id                                                                                                                       title  price  rating  similarity  price_numeric  rating_numeric
    4 B08FR4SRYS                                           Ezkrwxn Women Sport Running Shoes Fashion Casual Atheltic Walking Tennis Sneakers  39.90     4.2    0.667139          39.90             4.2
    6 B09VB9Z2P4                                M MGRNDL Womens Air Running Shoes Breathable Tennis Fashion Sneakers for Walking Jogging Gym  19.99     4.1    0.658777          19.99             4.1
    8 B09D3N4PRM                       RUNMAXX Running Shoes for Men Walking Shoes Non Slip Fashion Sneakers Gym Tennis Sport Athletic Shoes  22.99     4.3    0.648363          22.99             4.3
   17 B08K7FZ59Q Soulsfeng Running Shoes Low-top Lace-up Breathable Gym Trainer Tennis Walking Athletic Shoes Fashion Sneakers for Men Women  39.00     4.5    0.625051          39.00             4.5
   27

In [34]:
faiss.write_index(
    text_index,
    "../index/text_product_index.faiss"
)

print("Text FAISS index saved successfully.")

Text FAISS index saved successfully.


In [35]:
final_results.to_csv(
    "../data/processed/sample_recommendations.csv",
    index=False
)

print("Sample recommendations saved.")

Sample recommendations saved.
